# Reproducing Nair et al. (2018) from CRCNS data set bf-1

**Paper.** Nair J, Klaassen A-L, Arato J, Vyssotski AL, Harvey M, Rainer G (2018). Basal forebrain
contributes to default mode network regulation. *PNAS* 115:1352-1357.
[doi:10.1073/pnas.1712431115](https://doi.org/10.1073/pnas.1712431115). The claims file was written
from the version of record and its supplement (SI Results, SI Materials and Methods, legends of
Figs. S1 to S7).

**Data set.** CRCNS bf-1, "Basal forebrain LFP recordings from rats in home cage and during arena
exploration", [doi:10.6080/K0MK6B2Q](https://doi.org/10.6080/K0MK6B2Q), read from the public bucket
`crcnsarchive` under `bf-1/live/`. Version: `unversioned`. CRCNS data sets have no versions; the
paper cites the data set's DOI only, and the bucket holds one copy (`live`, no `old`), so that copy
is the one read. It is pinned by the MD5 sum of its single data file, from the data set's
`checksums.md5`: `basal_forebrain_lfp.tar.gz` a8acbf0400260f8395257778ed04dd8c (22,301,364
bytes); the notebook checks the sum of what it streams against it. The data descriptor is version
0.5 (Jan 23, 2018).

**What the data set holds.** BF LFP at 400 Hz from six rats, as 2-s segments with artifact segments
removed, one movement-sensor value per segment, and two home-cage and two arena sessions per rat.
It holds no ACC or VC recordings, no single units, no object sessions, no behaviour scoring or
video, no day labels, no identification of bipolar electrodes and no recording across the transfer
between arena and home cage.

| Claim | Tested here |
|---|---|
| fig3a-bf-gamma-reduction | tested |
| fig3a-each-hemisphere-significant | tested |
| fig3b-arena-gamma-unrelated-to-movement | tested |
| fig3b-home-cage-gamma-increases-with-movement | tested |
| fig3b-matched-movement | tested |
| fig3c-home-cage-above-both-sessions | tested (day factor dropped, as the criterion says) |
| fig3d-gamma-rises-with-time-in-arena | tested |
| figs2-bf-low-frequency-bands | tested |
| figs3-unimodal-states | tested |
| figs5-arena-excursions | tested |
| bipolar-gamma-reduction | blocked: metadata_insufficient (no rat is identified as bipolar) |
| fig3c-consistent-over-days | blocked: metadata_insufficient (no day labels) |
| fig3d-rapid-switch-to-home-cage | blocked: data_missing (no recording across the transfer) |
| behavior-arena-time-budget, behavior-home-cage-less-exploration, behavior-movement-speed, fig3f-grooming-quiet-arena, figs6-object-more-stationary | blocked: data_missing (no video or behaviour scoring) |
| fig4-object-exploration-suppressed | blocked: data_missing (no object sessions) |
| fig5a-phase-locked-neurons, fig5cd-pli-max-at-gamma, spike-width-phase-locked | blocked: data_missing (no single units) |
| fig6a-acc-gamma-home-cage, fig6a-vc-gamma-unchanged, fig6b-coherence, fig6c-granger-bf-to-acc, fig6d-trivariate-granger, figs2-acc-vc-low-frequency-bands | blocked: data_missing (no ACC or VC) |

Method details common to the tested claims are in the docstring of `bflfp.py`, with the passage of
the paper or the claims file each comes from.

In [ ]:
import time
import warnings

import diptest
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import bflfp

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sums = bflfp.checksums()
recordings, archive = bflfp.load_recordings()
print(f"data set {bflfp.DATASET_ID} ({bflfp.DATASET_VERSION}, doi:{bflfp.DATASET_DOI}): "
      f"{len(recordings)} files streamed in {time.time() - started:.0f} s")
print("checksums.md5 lists:", sums)
print("MD5 of the streamed archive:", archive["md5"], f"({archive['bytes']} bytes);",
      "matches checksums.md5:", archive["md5"] == sums[bflfp.ARCHIVE])

## Data check

In [ ]:
rows = []
for r in recordings:
    starts = r["t"][0]
    dt = np.diff(r["t"], axis=0)
    span = starts[-1] + 2.0
    rows.append({"rat": r["rat"], "condition": r["condition"], "session": r["session"],
                 "variables": ",".join(r["variables"]), "samples_per_segment": r["lfp"].shape[0],
                 "segments": r["lfp"].shape[1], "minutes_kept": r["lfp"].shape[1] * 2 / 60,
                 "first_start_s": starts[0], "last_end_s": span,
                 "removed_pct": 100 * (1 - r["lfp"].shape[1] * 2 / span),
                 "sample_dt_ms": f"{1000 * dt.min():.4f}-{1000 * dt.max():.4f}",
                 "segment_steps_s": ",".join(f"{x:g}" for x in np.unique(np.round(np.diff(starts), 3))[:5]),
                 "within_segment_contiguous": bool(np.allclose(r["t"][-1] - r["t"][0], 799 / 400)),
                 "lfp_min": r["lfp"].min(), "lfp_max": r["lfp"].max(), "lfp_sd": r["lfp"].std(),
                 "mov_min": r["mov"].min(), "mov_median": np.median(r["mov"]), "mov_max": r["mov"].max(),
                 "md5": r["md5"][:8]})
info = pd.DataFrame(rows)
print("Files by rat and condition:")
print(pd.crosstab(info.rat, info.condition).to_string())
print("\nVariables in every file:", info.variables.unique().tolist())
print("Condition labels in the file names:", sorted(info.condition.unique()))
print(info.drop(columns=["variables"]).round(2).to_string(index=False))
print("\nRemoved segments, %% of each file's span: %.1f to %.1f (SI: 1-4%% of epochs rejected)"
      % (info.removed_pct.min(), info.removed_pct.max()))

Every file holds `LFP_data`, `T_data` and `mov_data` with 800 samples per segment, and sample
intervals of exactly 2.5 ms (400 Hz), as the descriptor says. Segment starts step by multiples of
2 s, so segments are consecutive 2-s blocks with some removed, as the descriptor says. Each file's
clock starts at or near 0, so no two files share a clock: an arena file cannot be placed in time
relative to a home-cage file. The descriptor names the home-cage condition `Hcage`; the files use
`HCage`. Arena files span 8.8 to 13 min and keep 8.5 to 11.8 min of segments; home-cage files keep 8.7 to
31.6 min. The share of removed
segments (2.6% to 23.5% of a file's span) is larger than the SI's 1-4% of rejected epochs, so the
data set's cleaning differs from the paper's rule or removed more. The LFP has no stated unit; its
range (a few hundred, standard deviations of 60 to 115) is consistent with microvolts, the unit of
the paper's Figure 4 axis. The movement values have no unit (the descriptor says dimensionless); in
rats 4 and 5 they are about a quarter of those in the other rats, so they are only compared within
a rat.

In [ ]:
shared = bflfp.shared_segments(recordings)
print("Pairs of files that share identical LFP segments:")
print(shared.to_string(index=False) if len(shared) else "none")
by_key = {bflfp.key(r): r for r in recordings}
for _, row in shared.iterrows():
    a, b = by_key[row["first"]], by_key[row["second"]]
    pos = {a["lfp"][:, j].tobytes(): j for j in range(a["lfp"].shape[1])}
    pairs = [(pos.get(b["lfp"][:, j].tobytes()), j) for j in range(b["lfp"].shape[1])]
    matched = [(i, j) for i, j in pairs if i is not None]
    same_mov = np.mean([a["mov"][i] == b["mov"][j] for i, j in matched])
    print(f"{row['second']}: {len(matched)} of {b['lfp'].shape[1]} segments are segments of {row['first']} "
          f"({a['lfp'].shape[1]} segments), shifted by {row['time_offsets_s']} s; they cover "
          f"{a['t'][0, matched[0][0]]:.0f} to {a['t'][0, matched[-1][0]] + 2:.0f} s of it; "
          f"movement values equal on {100 * same_mov:.0f}% of the shared segments")
    lag = [np.mean([a["mov"][i + k] == b["mov"][j] for i, j in matched if 0 <= i + k < len(a["mov"])]) for k in (-1, 1)]
    print(f"  movement values equal at a shift of -1 segment: {100 * lag[0]:.0f}%, +1 segment: {100 * lag[1]:.0f}%")
DUPLICATES = {row["second"] for _, row in shared.iterrows()}
print("Left out as a second deposit of the same recording:", DUPLICATES)

Rat 3's home-cage session 2 is not a separate recording: all but 4 of its 291 segments are
segments of rat 3's home-cage session 1, 100 s later on that file's clock, with the same LFP. It is
counted once: the analysis uses session 1 and leaves session 2 out. Rat 3 so has one home-cage
recording. Where a test pairs sessions, the pair of rat 3's arena session 2 is dropped (11 pairs),
and the result with the files as delivered (12 pairs) is shown as a variant.

In [ ]:
# Spectra: line noise at 50 Hz (mains in Switzerland) lies inside the 30-80 Hz gamma band.
rows = []
for r in recordings:
    f, p = bflfp.spectrum(r["lfp"])
    mean = p.mean(axis=1)
    near = (f >= 45) & (f <= 55) & (f != 50)
    band = (f >= 20) & (f <= 100)
    rows.append({"rat": r["rat"], "condition": r["condition"], "session": r["session"],
                 "peak_20_100_Hz": f[band][np.argmax(mean[band])],
                 "power_50Hz_over_45_55": mean[f == 50][0] / mean[near].mean()})
lines = pd.DataFrame(rows)
print(lines.round(2).to_string(index=False))
print("Frequency resolution of a 2-s epoch: %.2f Hz; highest frequency %.0f Hz" % (f[1] - f[0], f[-1]))

The spectral peak between 20 and 100 Hz lies at 46.5 to 57.5 Hz in every home-cage file, the BF
gamma the paper describes; in arena files it lies at 50 to 61 Hz in half and at 20 to 30 Hz in the
others. One file, rat 2 home cage session 1, has a narrow line at 50 Hz, 26 times the
power of its neighbours; a few others have a smaller one (up to 3 times). The paper does not
mention line noise; the 30-80 Hz band includes 50 Hz, and the effect of leaving out 49-51 Hz is
shown under fig3a-bf-gamma-reduction.

In [ ]:
# Do the movement values share the clock of the LFP? Head movement adds broadband muscle and motion
# artefact to an LFP, seen best above the LFP bands (100-200 Hz here). Within each file the
# correlation of that power with the movement values should peak at zero shift of the movement
# values (shifts in segments, by position in the file); theta power (6-10 Hz) is shown alongside.
shifts = range(-5, 6)
rows = []
for r in recordings:
    f, p = bflfp.spectrum(r["lfp"])
    for name, (lo, hi) in (("100-200 Hz", (100, 200)), ("theta", (6, 10))):
        x, mv = np.log10(p[(f >= lo) & (f <= hi)].mean(axis=0)), r["mov"]
        rho = {k: (stats.spearmanr(x[k:], mv[:len(mv) - k])[0] if k >= 0 else stats.spearmanr(x[:k], mv[-k:])[0])
               for k in shifts}
        rows.append({"measure": name, "rat": r["rat"], "condition": r["condition"], "session": r["session"],
                     **{f"s{k:+d}": v for k, v in rho.items()}, "best_shift": max(rho, key=rho.get)})
sync = pd.DataFrame(rows)
print("Spearman correlation of log band power with movement; s+k pairs the LFP of segment i+k with the movement of segment i:")
print(sync.round(2).to_string(index=False))
for name, g in sync.groupby("measure"):
    print(f"\n{name}: best shift per file {g.best_shift.value_counts().sort_index().to_dict()}; median correlation "
          f"at 0 {g['s+0'].median():.2f}, at +1 {g['s+1'].median():.2f}, at -1 {g['s-1'].median():.2f}, "
          f"at +-5 {np.median(np.r_[g['s+5'], g['s-5']]):.2f}")

In [ ]:
# Which way does movement go with gamma and theta power, by condition (epochs, per file)?
tab_all = bflfp.epoch_table(recordings, 2, "asis")
rows = []
for (rat, cond, sess), g in tab_all.groupby(["rat", "condition", "session"]):
    rows.append({"rat": rat, "condition": cond, "session": sess,
                 "rho_mov_gamma": stats.spearmanr(g.mov, g.gamma)[0],
                 "rho_mov_theta": stats.spearmanr(g.mov, g.theta)[0]})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("Median movement value by condition, per rat:")
print(tab_all.groupby(["rat", "condition"]).mov.median().unstack().round(0).to_string())

In 23 of 24 files the correlation of 100-200 Hz power with the movement values is largest at a shift
of 0 or +1 segment and falls off at larger shifts (printed above), so the movement values are
aligned with the LFP segments to within one segment (2 s). In 13 files the peak is at +1: the
movement value of a segment goes best with the LFP of the next segment, so the movement stream may
lead the LFP by up to one segment; this is consistent with the data and not shown to be an offset.
Theta power gives a weaker and less regular relation. Movement is higher in the arena than in the
home cage in every rat, as the conditions imply. This is consistent with the labels and the
movement stream, and does not by itself prove each file's label. No two files share a clock, so
alignment across files cannot be checked.

## Units of analysis and readings shared by the claims

Gamma power and the other band powers are computed per 2-s segment (`bflfp.epoch_table`). The
claims file's header names readings that apply to several claims, and each is run:

- epoch length: 2-s segments (main) or 10-s epochs of five contiguous segments;
- artifacts: segments as delivered (main) or the SI's peak-to-peak rule applied again;
- unit of pairing: per rat, the two sessions of a condition averaged (rat 3's home cage is session 1
  alone), n = 6; or session pairs (session 1 with session 1), n = 11 without the duplicate, and 12
  with the files as delivered.

In [ ]:
def drop_duplicates(tab):
    keys = list(zip(tab.rat, tab.condition, tab.session))
    return tab[[k not in DUPLICATES for k in keys]].copy()


def unit_means(tab, unit, measure="gamma"):
    # Per recording, the mean over its epochs; then per rat (sessions averaged) or per session pair.
    m = tab.groupby(["rat", "condition", "session"])[measure].mean().reset_index()
    if unit != "session_asis":
        m = m[[(a, b, c) not in DUPLICATES for a, b, c in zip(m.rat, m.condition, m.session)]]
    if unit == "rat":
        return m.groupby(["rat", "condition"])[measure].mean().unstack()
    return m.set_index(["rat", "session", "condition"])[measure].unstack().dropna()


tables = {(ep, art): bflfp.epoch_table(recordings, ep, art) for ep in (2, 10) for art in ("asis", "ptp")}
for (ep, art), t in tables.items():
    kept = t.groupby(["rat", "condition", "session"]).size()
    print(f"{ep}-s epochs, artifacts {art}: {len(t)} epochs, per file {kept.min()} to {kept.max()}")
n2 = sum(r["lfp"].shape[1] for r in recordings)
print(f"The peak-to-peak rule drops {n2 - len(tables[(2, 'ptp')])} of {n2} segments "
      f"({100 * (1 - len(tables[(2, 'ptp')]) / n2):.1f}%); the SI reports 1-4%")
main = drop_duplicates(tables[(2, "asis")])

## fig3a-bf-gamma-reduction

> BF gamma power was reduced by 35 +/- 3% when rats moved from the home cage to the novel arena, a
> highly significant reduction. (Expected 35 +/- 3%; criterion: reduction within tolerance under
> reading (a) mean over recordings of 100 x (1 - arena / home cage) or (b) from the group means,
> and a paired t test over rats at p < 0.05 with home cage higher; partially reproduced when the
> test holds and no reading is within tolerance.)

In [ ]:
rows = []
for (ep, art), t in tables.items():
    for unit in ("rat", "session", "session_asis"):
        w = unit_means(t, unit)
        h, a = w["HCage"], w["Arena"]
        test = stats.ttest_rel(h, a)
        rows.append({"epoch_s": ep, "artifacts": art, "unit": unit, "n": len(w),
                     "reading_a_pct": (100 * (1 - a / h)).mean(), "reading_b_pct": 100 * (1 - a.mean() / h.mean()),
                     "sem_a": stats.sem(100 * (1 - a / h)), "t": test.statistic, "p": test.pvalue,
                     "home_higher_in": f"{int((h > a).sum())}/{len(w)}"})
grid = pd.DataFrame(rows)
print(grid.round(4).to_string(index=False))
print("\nReadings with the reduction within 32-38%:",
      int(((grid.reading_a_pct.between(32, 38)) | (grid.reading_b_pct.between(32, 38))).sum()), "of", len(grid))
w = unit_means(main, "rat")
print("\nPer rat (main reading: 2-s, as delivered), gamma power (uV^2/Hz if the LFP is in uV):")
print(w.assign(reduction_pct=100 * (1 - w.Arena / w.HCage)).round(1).to_string())

In [ ]:
# Variants not named by the criterion: 49-51 Hz left out of the band (line noise), and the amplitude
# spectrum |FFT| in place of power, which the data descriptor's example uses (abs(fft(...))).
rows = []
for r in recordings:
    if bflfp.key(r) in DUPLICATES:
        continue
    f, p = bflfp.spectrum(r["lfp"])
    band = (f >= 30) & (f <= 80)
    notch = band & ~((f >= 49) & (f <= 51))
    rows.append({"rat": r["rat"], "condition": r["condition"], "session": r["session"],
                 "power": p[band].mean(), "power_no_50": p[notch].mean(), "amplitude": np.sqrt(p[band]).mean()})
var = pd.DataFrame(rows)
for measure in ("power", "power_no_50", "amplitude"):
    w = var.groupby(["rat", "condition"])[measure].mean().unstack()
    h, a = w["HCage"], w["Arena"]
    print(f"{measure:12s} reduction (a) {(100 * (1 - a / h)).mean():.1f}%  (b) {100 * (1 - a.mean() / h.mean()):.1f}%  "
          f"paired t p = {stats.ttest_rel(h, a).pvalue:.3g}, n = {len(w)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for r in recordings:
    if bflfp.key(r) in DUPLICATES:
        continue
    f, p = bflfp.spectrum(r["lfp"])
    keep = (f >= 1) & (f <= 100)
    ax[0].semilogy(f[keep], p[keep].mean(axis=1), color="C0" if r["condition"] == "HCage" else "C3", lw=0.7)
ax[0].set(xlabel="frequency (Hz)", ylabel="power (LFP unit^2/Hz)", title="Mean spectrum per file (blue home cage, red arena)")
w = unit_means(main, "rat")
for rat, row in w.iterrows():
    ax[1].plot([0, 1], [row.HCage, row.Arena], "o-", color="0.4")
ax[1].set(xticks=[0, 1], xticklabels=["home cage", "arena"], ylabel="gamma power", title="Per rat")
plt.tight_layout()
plt.show()
print("Plotted gamma power per rat ranges from %.1f to %.1f" % (w.values.min(), w.values.max()))

**Result: partially reproduced.** Home-cage gamma exceeds arena gamma in every rat and every
session pair, and the paired t test gives p = 0.012 over 6 rats (p = 0.00047 over 11 session pairs).
The reduction is 48% under both percentage readings and every epoch, artifact and pairing reading
(46.8% to 48.1%), outside 35 +/- 3%. Leaving out 49-51 Hz does not change this; the amplitude
spectrum, which the paper does not use, gives a smaller reduction (printed above).

## fig3a-each-hemisphere-significant

> The BF gamma reduction in the arena was significant in every one of the 22 hemispheres analyzed
> individually. (Criterion: per recording, a rat's arena session against its home-cage session of
> the same number, unpaired t test of epoch gamma, every recording p < 0.01 with home cage higher.)

In [ ]:
rows = []
for ep_art, t in (("2-s, as delivered", tables[(2, "asis")]), ("2-s, peak-to-peak rule", tables[(2, "ptp")]),
                  ("10-s, as delivered", tables[(10, "asis")])):
    for (rat, sess), g in t[t.condition == "Arena"].groupby(["rat", "session"]):
        for home_session, label in ((sess, "same number"), (1, "home session 1")):
            if label == "home session 1" and not (rat == 3 and sess == 2):
                continue
            h = t[(t.rat == rat) & (t.condition == "HCage") & (t.session == home_session)]
            test = stats.ttest_ind(h.gamma, g.gamma)
            rows.append({"epochs": ep_art, "rat": rat, "session": sess, "home_cage_file": f"session {home_session}",
                         "n_home": len(h), "n_arena": len(g), "home_mean": h.gamma.mean(), "arena_mean": g.gamma.mean(),
                         "t": test.statistic, "p": test.pvalue})
each = pd.DataFrame(rows)
print(each.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for ep_art, g in each.groupby("epochs", sort=False):
    ok = (g.p < 0.01) & (g.home_mean > g.arena_mean)
    print(f"{ep_art}: {int(ok.sum())} of {len(g)} comparisons p < 0.01 with home cage higher; largest p = {g.p.max():.2g}")

**Result: reproduced.** Each of the 12 arena recordings has lower gamma than its home-cage
recording at p < 0.01 by the unpaired t test over epochs, under every epoch reading, including
rat 3's arena session 2 against either of its home-cage files (largest p printed above). The unit
is a rat and session, not a hemisphere.

## fig3b-arena-gamma-unrelated-to-movement

> In the arena, BF gamma power did not vary with movement-sensor quartile. (Criterion, absence:
> 2-s epochs in quartiles of the movement value within each condition, within each recording or
> over a rat's pooled sessions; one-way ANOVA over quartiles on per-rat quartile means; reproduced
> when p > 0.1, not reproduced when p < 0.05. Variants: the ANOVA over epochs, and a
> repeated-measures ANOVA.)

In [ ]:
def quartile_means(tab, scope):
    # Quartiles of the movement value within each recording, or within a rat's pooled sessions,
    # for each condition separately (criterion). Returns per-rat quartile means and the epoch table.
    grp = ["rat", "condition", "session"] if scope == "recording" else ["rat", "condition"]
    t = tab.copy()
    t["q"] = t.groupby(grp)["mov"].transform(lambda x: pd.qcut(x, 4, labels=False) + 1)
    return t.groupby(["rat", "condition", "q"])[["gamma", "mov"]].mean().reset_index(), t


def quartile_tests(tab, cond):
    rows = []
    for scope in ("recording", "rat"):
        qm, t = quartile_means(tab, scope)
        d = qm[qm.condition == cond]
        oneway = stats.f_oneway(*[d[d.q == q].gamma for q in range(1, 5)])
        epochs = stats.f_oneway(*[t[(t.condition == cond) & (t.q == q)].gamma for q in range(1, 5)])
        rm = AnovaRM(d, "gamma", "rat", within=["q"]).fit().anova_table
        means = d.groupby("q").gamma.mean()
        # effect size: eta squared of the one-way ANOVA on per-rat means
        grand = d.gamma.mean()
        eta2 = sum(len(d[d.q == q]) * (means[q] - grand) ** 2 for q in range(1, 5)) / ((d.gamma - grand) ** 2).sum()
        rows.append({"quartiles_within": scope, "Q1": means[1], "Q2": means[2], "Q3": means[3], "Q4": means[4],
                     "Q4_over_Q1": means[4] / means[1], "rats_Q4_gt_Q1": int((d[d.q == 4].gamma.values > d[d.q == 1].gamma.values).sum()),
                     "oneway_F": oneway.statistic, "oneway_p": oneway.pvalue, "eta2": eta2,
                     "epochs_F": epochs.statistic, "epochs_p": epochs.pvalue,
                     "rm_F": rm["F Value"].iloc[0], "rm_p": rm["Pr > F"].iloc[0]})
    return pd.DataFrame(rows)


print("Arena, 2-s epochs as delivered:")
print(quartile_tests(main, "Arena").to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("\nArena, other epoch and artifact readings:")
for (ep, art), t in tables.items():
    if (ep, art) == (2, "asis"):
        continue
    r = quartile_tests(drop_duplicates(t), "Arena")
    print(f"  {ep}-s, {art}: one-way p = {r.oneway_p.round(3).tolist()}, Q4/Q1 = {r.Q4_over_Q1.round(3).tolist()}")
qm, _ = quartile_means(main, "recording")
print("\nPer-rat quartile means, arena (quartiles within each recording):")
print(qm[qm.condition == "Arena"].pivot(index="rat", columns="q", values="gamma").round(1).to_string())

**Result: reproduced.** In the arena the per-rat quartile means of gamma differ by a few percent
(Q4/Q1 near 1), and the one-way ANOVA over quartiles gives p > 0.1 with quartiles formed within
each recording or within each rat, under every epoch and artifact reading. The epoch-level and
repeated-measures variants also give p > 0.1 (printed above).

## fig3b-home-cage-gamma-increases-with-movement

> In the home cage, BF gamma power increased with movement-sensor quartile. (Criterion: per-rat
> quartile means as for the arena claim; reproduced when the one-way ANOVA over quartiles gives
> p < 0.05 and the fourth quartile mean exceeds the first.)

In [ ]:
print("Home cage, 2-s epochs as delivered:")
print(quartile_tests(main, "HCage").to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("\nHome cage, other epoch and artifact readings:")
for (ep, art), t in tables.items():
    if (ep, art) == (2, "asis"):
        continue
    r = quartile_tests(drop_duplicates(t), "HCage")
    print(f"  {ep}-s, {art}: one-way p = {r.oneway_p.round(3).tolist()}, rm p = {r.rm_p.round(6).tolist()}, "
          f"Q4/Q1 = {r.Q4_over_Q1.round(2).tolist()}")
print("\nPer-rat quartile means, home cage (quartiles within each recording):")
print(qm[qm.condition == "HCage"].pivot(index="rat", columns="q", values="gamma").round(1).to_string())
print("\nBetween-rat spread of home-cage gamma (per-rat means): %.1f to %.1f"
      % (unit_means(main, "rat").HCage.min(), unit_means(main, "rat").HCage.max()))

In [ ]:
# Variant: the data check found the movement value of a segment may go with the LFP of the next
# segment. Pair each LFP segment with the movement value of the segment before it in its file.
shifted = main.copy()
shifted["mov"] = shifted.groupby(["rat", "condition", "session"]).mov.shift(1)
shifted = shifted.dropna(subset=["mov"])
for cond in ("Arena", "HCage"):
    r = quartile_tests(shifted, cond)
    print(f"{cond}, movement shifted by one segment: one-way p = {r.oneway_p.round(3).tolist()}, "
          f"rm p = {r.rm_p.round(6).tolist()}, Q4/Q1 = {r.Q4_over_Q1.round(2).tolist()}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for i, cond in enumerate(("HCage", "Arena")):
    d = qm[qm.condition == cond]
    for rat, g in d.groupby("rat"):
        ax[i].plot(g.q, g.gamma, "o-", color="0.5", lw=0.8)
    ax[i].plot(range(1, 5), d.groupby("q").gamma.mean(), "k-", lw=2)
    ax[i].set(title=cond, xlabel="movement quartile", xticks=range(1, 5))
ax[0].set_ylabel("gamma power")
plt.tight_layout()
plt.show()
print("Plotted values range from %.1f to %.1f" % (qm.gamma.min(), qm.gamma.max()))

**Result: not reproduced.** Home-cage gamma rises from the first to the fourth movement quartile in
every rat (Q4/Q1 about 1.45), but the criterion's test, a one-way ANOVA over quartiles on the
per-rat quartile means, gives p of about 0.8, because gamma differs between rats several-fold. The
variants that take the rat into account find the rise: the repeated-measures ANOVA gives p < 0.001
and the ANOVA over epochs p < 1e-40 (printed above). Pairing each segment with the movement value
of the segment before it, a variant, gives the same picture (one-way p 0.75 to 0.79).

## fig3b-matched-movement

> Gamma suppression in the arena remained highly significant when epochs matched for movement were
> compared, for example the fourth home-cage quartile against the third arena quartile.
> (Criterion: reading (a) that pair; (b) the pair of quartiles, one per condition, whose mean
> movement values are closest; one-way ANOVA of the per-rat gamma means of the two groups,
> p < 0.05 with the home cage higher.)

In [ ]:
rows = []
for scope in ("recording", "rat"):
    qm_s, _ = quartile_means(main, scope)
    mov = qm_s.groupby(["condition", "q"]).mov.mean().unstack()
    diff = pd.DataFrame([(h, a, abs(mov.loc["HCage", h] - mov.loc["Arena", a])) for h in range(1, 5) for a in range(1, 5)],
                        columns=["home_q", "arena_q", "mov_diff"])
    closest = diff.sort_values("mov_diff").iloc[0]
    for reading, (hq, aq) in (("(a) home Q4 vs arena Q3", (4, 3)),
                              ("(b) closest movement", (int(closest.home_q), int(closest.arena_q)))):
        h = qm_s[(qm_s.condition == "HCage") & (qm_s.q == hq)].set_index("rat")
        a = qm_s[(qm_s.condition == "Arena") & (qm_s.q == aq)].set_index("rat")
        oneway = stats.f_oneway(h.gamma, a.gamma)
        paired = stats.ttest_rel(h.gamma, a.gamma)
        rows.append({"quartiles_within": scope, "reading": reading, "home_q": hq, "arena_q": aq,
                     "home_mov": h.mov.mean(), "arena_mov": a.mov.mean(), "home_gamma": h.gamma.mean(),
                     "arena_gamma": a.gamma.mean(), "rats_home_higher": int((h.gamma > a.gamma).sum()),
                     "oneway_p": oneway.pvalue, "paired_t_p_variant": paired.pvalue})
    if scope == "recording":
        print("Mean movement value per quartile (mean over rats), quartiles within each recording:")
        print(mov.round(0).to_string())
matched = pd.DataFrame(rows)
print(matched.to_string(index=False, float_format=lambda x: f"{x:.3g}"))

# Variant: movement matched within each rat, the home-cage and arena quartiles closest in that rat.
rows = []
qm_r, _ = quartile_means(main, "recording")
for rat, d in qm_r.groupby("rat"):
    h, a = d[d.condition == "HCage"].set_index("q"), d[d.condition == "Arena"].set_index("q")
    hq, aq = min(((x, y) for x in range(1, 5) for y in range(1, 5)), key=lambda p: abs(h.mov[p[0]] - a.mov[p[1]]))
    rows.append({"rat": rat, "home_q": hq, "arena_q": aq, "home_mov": h.mov[hq], "arena_mov": a.mov[aq],
                 "home_gamma": h.gamma[hq], "arena_gamma": a.gamma[aq]})
within = pd.DataFrame(rows)
print("\nVariant, quartiles matched within each rat:")
print(within.round(1).to_string(index=False))
print("one-way p = %.3g, paired t p = %.3g" % (stats.f_oneway(within.home_gamma, within.arena_gamma).pvalue,
                                            stats.ttest_rel(within.home_gamma, within.arena_gamma).pvalue))

**Result: not reproduced.** Under both readings the movement-matched home-cage gamma exceeds the
arena gamma in every rat, by about a factor of two, but the criterion's one-way ANOVA of the per-rat
means of the two groups gives p of about 0.07 (printed above), above 0.05. A paired test of the
same values, a variant, gives p < 0.05.

## fig3c-home-cage-above-both-sessions

> BF gamma was larger in the home cage than in either of the two daily arena sessions, and slightly
> larger (5.62 +/- 1.2%) in the second arena session than in the first, with no interaction
> between day and session. (Criterion: levels home cage, arena 1, arena 2; Tukey's HSD, with
> Bonferroni-corrected paired t tests as the variant; increase as 100 x (arena 2 / arena 1 - 1)
> averaged over recordings, or from the group means. Without day labels the two arena sessions are
> compared within rat by a paired t test and the day factor is dropped.)

In [ ]:
for label, t in (("2-s, as delivered", main), ("2-s, peak-to-peak rule", drop_duplicates(tables[(2, "ptp")])),
                 ("10-s, as delivered", drop_duplicates(tables[(10, "asis")]))):
    m = t.groupby(["rat", "condition", "session"]).gamma.mean()
    home = m.xs("HCage", level="condition").groupby("rat").mean()
    a1 = m.xs("Arena", level="condition").xs(1, level="session")
    a2 = m.xs("Arena", level="condition").xs(2, level="session")
    levels = pd.DataFrame({"home": home, "arena1": a1, "arena2": a2})
    print(f"--- {label}")
    if label.startswith("2-s, as"):
        print(levels.round(1).to_string())
    tukey = pairwise_tukeyhsd(np.r_[home, a1, a2], ["home"] * 6 + ["arena1"] * 6 + ["arena2"] * 6)
    print(tukey.summary())
    for x, y, name in ((home, a1, "home vs arena1"), (home, a2, "home vs arena2"), (a2, a1, "arena2 vs arena1")):
        p = stats.ttest_rel(x, y).pvalue
        print(f"paired t {name}: mean difference {np.mean(x - y):.2f}, p = {p:.3g}, Bonferroni (x3) p = {min(1, 3 * p):.3g}")
    inc = 100 * (a2 / a1 - 1)
    print("increase arena2 over arena1: (a) mean over rats %.2f%% (per rat %s); (b) group means %.2f%%"
          % (inc.mean(), inc.round(1).tolist(), 100 * (a2.mean() / a1.mean() - 1)))

**Result: not reproduced.** Arena session 2 has lower gamma than arena session 1 in five of six
rats, a mean change of -15.9% (group means -15.7%), where the paper reports +5.62%; the paired t test
of session 2 against session 1 gives p = 0.020 in the opposite direction. The home cage exceeds both
arena sessions by Bonferroni-corrected paired t tests (p of about 0.04), but not by Tukey's HSD on
the per-rat values (p 0.19 and 0.30). The outcome rests on the assumption that the data set's
session numbers 1 and 2 are the paper's first and second daily arena sessions.

## fig3d-gamma-rises-with-time-in-arena

> Within arena sessions BF gamma stayed suppressed but rose with time spent in the arena (r = 0.72,
> n = 22). (Expected r = 0.72 +/- 0.072; criterion: reading (a) Pearson r between time and gamma
> averaged across recordings in 1-min bins over the 10-min session, with 2-s epochs as the variant;
> (b) the mean over recordings of each recording's r; (c) r over all epochs pooled; reproduced when
> under one reading r is within tolerance and positive at p < 0.05.)

In [ ]:
arena = main[main.condition == "Arena"].copy()
arena["minute"] = (arena.t // 60).astype(int)
rows = []
for window, a in (("first 10 min", arena[arena.t < 600]), ("whole file", arena)):
    per = a.groupby(["rat", "session", "minute"]).gamma.mean().reset_index()
    curve = per.groupby("minute").gamma.mean()
    r_a, p_a = stats.pearsonr(curve.index, curve.values)
    per_epoch = a.groupby("t").gamma.mean()
    r_a2, p_a2 = stats.pearsonr(per_epoch.index, per_epoch.values)
    rs = np.array([stats.pearsonr(g.t, g.gamma)[0] for _, g in a.groupby(["rat", "session"])])
    t_b = stats.ttest_1samp(rs, 0)
    r_c, p_c = stats.pearsonr(a.t, a.gamma)
    rows += [{"window": window, "reading": "(a) 1-min bins, mean over recordings", "r": r_a, "p": p_a, "n": len(curve)},
             {"window": window, "reading": "(a) variant, 2-s epochs, mean over recordings", "r": r_a2, "p": p_a2, "n": len(per_epoch)},
             {"window": window, "reading": "(b) mean of per-recording r", "r": rs.mean(), "p": t_b.pvalue, "n": len(rs)},
             {"window": window, "reading": "(c) all epochs pooled", "r": r_c, "p": p_c, "n": len(a)}]
    if window == "first 10 min":
        print("Gamma per minute, mean over the 12 arena recordings:", curve.round(1).tolist())
        print("Per-recording r:", np.round(rs, 2).tolist())
        print("Recordings contributing to each minute:", per.groupby("minute").size().tolist())
    else:
        print("Whole file, recordings contributing to each minute:", per.groupby("minute").size().tolist())
corr = pd.DataFrame(rows)
print(corr.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("Readings within 0.648-0.792 and positive at p < 0.05:",
      corr[(corr.r.between(0.648, 0.792)) & (corr.p < 0.05)][["window", "reading"]].values.tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
per = arena[arena.t < 600].groupby(["rat", "session", "minute"]).gamma.mean().reset_index()
for _, g in per.groupby(["rat", "session"]):
    ax.plot(g.minute + 0.5, g.gamma / g.gamma.mean(), color="0.7", lw=0.8)
ax.plot(np.arange(10) + 0.5, per.groupby("minute").gamma.mean() / per.gamma.mean(), "k-", lw=2)
ax.set(xlabel="time in arena (min)", ylabel="gamma / recording mean", title="Arena gamma by minute")
plt.show()
print("Plotted ratios range from %.2f to %.2f" % ((per.gamma / per.groupby(["rat", "session"]).gamma.transform("mean")).min(),
                                                 (per.gamma / per.groupby(["rat", "session"]).gamma.transform("mean")).max()))

**Result: partially reproduced.** Gamma rises with time in the arena: under reading (a), over the
first 10 min in 1-min bins, r = 0.86 (p = 0.0015), above the tolerance of 0.72 +/- 0.072; reading
(b) gives a mean per-recording r of 0.13 (p = 0.056) and reading (c) r = 0.045 (p = 0.009). No
reading puts r within tolerance, while the positive correlation holds at p < 0.05 under (a) and
(c).

## figs2-bf-low-frequency-bands

> In the BF, delta and beta power, like gamma, were higher in the home cage than in the arena,
> while theta power was strongly suppressed in the home cage. (Criterion: band power per rat and
> condition, paired t test per band; reproduced when all four bands differ at p < 0.05 in the
> stated directions; partially when theta and gamma do and one of delta and beta does not.)

In [ ]:
expected_sign = {"delta": 1, "theta": -1, "beta": 1, "gamma": 1}
rows = []
for (ep, art), t in tables.items():
    for unit in ("rat", "session", "session_asis"):
        for band, sign in expected_sign.items():
            w = unit_means(t, unit, band)
            test = stats.ttest_rel(w.HCage, w.Arena)
            rows.append({"epoch_s": ep, "artifacts": art, "unit": unit, "band": band, "n": len(w),
                         "home": w.HCage.mean(), "arena": w.Arena.mean(), "home_over_arena": (w.HCage / w.Arena).mean(),
                         "t": test.statistic, "p": test.pvalue,
                         "holds": bool(test.pvalue < 0.05 and np.sign(test.statistic) == sign)})
bands = pd.DataFrame(rows)
print(bands[(bands.epoch_s == 2) & (bands.artifacts == "asis")].to_string(index=False, float_format=lambda x: f"{x:.3g}"))
summary = bands.groupby(["epoch_s", "artifacts", "unit"]).holds.agg(lambda x: "".join("Y" if v else "n" for v in x))
print("\nBands holding (delta, theta, beta, gamma) under every reading:")
print(summary.to_string())

**Result: reproduced.** Over the six rats, delta, beta and gamma power are higher in the home cage
and theta lower, each at p < 0.05 by the paired t test (theta p < 0.001), and all four hold under
every epoch, artifact and pairing reading (printed above).

## figs3-unimodal-states

> Within each condition, the distributions of BF theta and gamma power over 2-s epochs were
> unimodal, with no evidence of substates, in each of six rats. (Criterion, absence: Hartigan dip
> test per rat, condition and band; reproduced when every test gives p > 0.1, not reproduced when
> any gives p < 0.05; readings (a) a rat's two sessions of a condition pooled, (b) each session
> alone.)

In [ ]:
rows = []
for reading in ("(a) sessions pooled", "(b) each session"):
    groups = main.groupby(["rat", "condition"]) if reading.startswith("(a)") else main.groupby(["rat", "condition", "session"])
    for k, g in groups:
        for band in ("theta", "gamma"):
            dip, p = diptest.diptest(g[band].values)
            rows.append({"reading": reading, "group": k, "band": band, "n_epochs": len(g),
                         "minutes": len(g) * 2 / 60, "dip": dip, "p": p})
dips = pd.DataFrame(rows)
print(dips.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for reading, g in dips.groupby("reading"):
    print(f"{reading}: {len(g)} tests, smallest p = {g.p.min():.3f}, tests with p <= 0.1: {int((g.p <= 0.1).sum())}, "
          f"with p < 0.05: {int((g.p < 0.05).sum())}")
# Variant: the duplicate file kept (rat 3 home cage session 2).
dup = tables[(2, "asis")]
for k in DUPLICATES:
    g = dup[(dup.rat == k[0]) & (dup.condition == k[1]) & (dup.session == k[2])]
    print("Variant, the left-out file", k, {b: round(diptest.diptest(g[b].values)[1], 3) for b in ("theta", "gamma")})

**Result: reproduced.** Every Hartigan dip test of theta and gamma power over 2-s epochs, per rat
and condition, gives p > 0.1, with sessions pooled (24 tests) or taken alone (46 tests; rat 3 has one home-cage file); the
smallest p is printed above. The p-values come from the `diptest` package's tabulated null
distribution.

## figs5-arena-excursions

> During about 20 min of arena exploration, BF gamma made on average eight brief excursions into the
> home-cage range, lasting 8 to 44 s (16 +/- 8 s), with a clear rise in gamma power (0.56 +/- 0.27)
> and no change in theta power (-0.01 +/- 0.82) relative to 4 to 8 s before the peak. (Criterion:
> boxcar of 3, threshold at the intersection of the home-cage and arena gamma histograms, readings
> for the bins (20 equal bins; kernel density), the power scale (raw, log10, normalized to the rat's
> mean) and the arena data (two sessions together; each alone, count doubled). Reproduced when, under
> one reading, count and mean duration are within tolerance, durations within 8 to 44 s widened by
> 10%, the gamma rise is significant (p < 0.05) and the theta change is not (p > 0.1); partially
> reproduced when the gamma and theta tests hold and the count or duration does not.)

In [ ]:
def scaled(x, scale, ref):
    return {"raw": x, "log10": np.log10(x), "normalized": x / ref}[scale]


def run_excursions(scale, method, data):
    per_rat, events = [], []
    for rat in range(1, 7):
        t = main[main.rat == rat]
        ref_g, ref_t = t.gamma.mean(), t.theta.mean()
        home, ar = t[t.condition == "HCage"], t[t.condition == "Arena"]
        if data == "together":
            sets = [(home, ar, 1)]
        else:
            sets = [(home[home.session == s] if (home.session == s).any() else home, ar[ar.session == s], 2) for s in (1, 2)]
        for h, a, factor in sets:
            find = bflfp.histogram_intersection if method == "20 bins" else bflfp.kde_intersection
            thr = find(scaled(h.gamma.values, scale, ref_g), scaled(a.gamma.values, scale, ref_g))
            ev = []
            for s, g in a.groupby("session"):  # files do not share a clock: one series per file
                ev += bflfp.excursions(scaled(g.gamma.values, scale, ref_g), scaled(g.theta.values, scale, ref_t),
                                       g.t.values, thr)
            per_rat.append({"rat": rat, "threshold": thr, "count_20min": factor * len(ev),
                            "arena_minutes": len(a) * 2 / 60})
            events += [dict(e, rat=rat) for e in ev]
    per_rat, events = pd.DataFrame(per_rat), pd.DataFrame(events)
    ok = events.dropna(subset=["gamma_before"])
    by_rat = ok.groupby("rat")[["gamma_peak", "gamma_before", "theta_peak", "theta_before"]].mean()
    g_test = stats.ttest_rel(by_rat.gamma_peak, by_rat.gamma_before)
    t_test = stats.ttest_rel(by_rat.theta_peak, by_rat.theta_before)
    return per_rat, events, {
        "scale": scale, "bins": method, "arena_data": data,
        "threshold_range": f"{per_rat.threshold.min():.3g}-{per_rat.threshold.max():.3g}",
        "count_per_20min": per_rat.groupby("rat").count_20min.mean().mean(),
        "mean_duration_s": events.duration_s.mean(), "sd_duration_s": events.duration_s.std(),
        "min_duration_s": events.duration_s.min(), "max_duration_s": events.duration_s.max(),
        "n_events": len(events), "events_without_before": len(events) - len(ok),
        "gamma_rise": (by_rat.gamma_peak - by_rat.gamma_before).mean(), "gamma_p": g_test.pvalue,
        "theta_change": (by_rat.theta_peak - by_rat.theta_before).mean(), "theta_p": t_test.pvalue}


results = []
for scale in ("raw", "log10", "normalized"):
    for method in ("20 bins", "kernel density"):
        for data in ("together", "each alone"):
            results.append(run_excursions(scale, method, data)[2])
exc = pd.DataFrame(results)
exc["count_ok"] = exc.count_per_20min.between(7.2, 8.8)
exc["duration_ok"] = exc.mean_duration_s.between(8, 24) & (exc.min_duration_s >= 7.2) & (exc.max_duration_s <= 48.4)
exc["gamma_ok"] = exc.gamma_p < 0.05
exc["theta_ok"] = exc.theta_p > 0.1
print(exc.to_string(index=False, float_format=lambda x: f"{x:.3g}"))

In [ ]:
per_rat, events, _ = run_excursions("log10", "20 bins", "together")
print("log10 scale, 20 bins, sessions together: per rat")
print(per_rat.round(3).to_string(index=False))
print("Durations (s), counts:", events.duration_s.value_counts().sort_index().to_dict())
print("Excursions lasting 8 s or more: %d of %d; their mean count per rat %.1f"
      % ((events.duration_s >= 8).sum(), len(events), (events.duration_s >= 8).groupby(events.rat).sum().mean()))
fig, ax = plt.subplots(figsize=(9, 3))
t = main[(main.rat == 1) & (main.condition == "Arena") & (main.session == 1)]
ax.plot(t.t / 60, bflfp.boxcar(np.log10(t.gamma.values)), lw=0.8)
ax.axhline(per_rat[per_rat.rat == 1].threshold.iloc[0], color="k", ls=":")
ax.set(xlabel="time (min)", ylabel="log10 gamma, boxcar 3", title="Rat 1, arena session 1")
plt.show()
print("Plotted range: %.2f to %.2f" % (bflfp.boxcar(np.log10(t.gamma.values)).min(), bflfp.boxcar(np.log10(t.gamma.values)).max()))

**Result: partially reproduced.** Gamma rises at the excursion peak under all 12 readings
(paired t test over rats, p from 0.0001 to 0.04), and the theta change is not significant
(p > 0.1) under 10 of them, among them every log10 reading; under the raw and normalized scales with
20 bins and the sessions taken together the theta test gives p < 0.05. No reading gives the paper's
count or durations: 10.3 to 14.3 excursions per 20 min in place of 8, mean durations of 5.1 to 5.7 s
in place of 16, and durations from 2 s, below the paper's shortest of 8 s. No reading puts the
threshold near 2.6, so the 0.56 and -0.01 values are reported and not judged.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig3a-bf-gamma-reduction | partially reproduced (48% reduction, p = 0.012 over 6 rats) |
| fig3a-each-hemisphere-significant | reproduced (12 of 12 recordings p < 0.01) |
| fig3b-arena-gamma-unrelated-to-movement | reproduced |
| fig3b-home-cage-gamma-increases-with-movement | not reproduced by the criterion's test (rise in every rat; repeated-measures p < 0.001) |
| fig3b-matched-movement | not reproduced by the criterion's test (home higher in every rat; one-way p about 0.07) |
| fig3c-home-cage-above-both-sessions | not reproduced (arena 2 lower than arena 1), assumed session numbering |
| fig3d-gamma-rises-with-time-in-arena | partially reproduced (r = 0.86 in 1-min bins) |
| figs2-bf-low-frequency-bands | reproduced |
| figs3-unimodal-states | reproduced |
| figs5-arena-excursions | partially reproduced (more and shorter excursions) |
| bipolar-gamma-reduction, fig3c-consistent-over-days | blocked, metadata insufficient |
| fig3d-rapid-switch-to-home-cage and 16 claims on behaviour, objects, units, ACC and VC | blocked, data missing |

How the analysis differs from the paper's: the data set gives one BF signal per rat for six rats,
so tests run over rats (n = 6) or session pairs, where the paper used 22 hemispheres; the segments
come already cleaned and down-sampled; the gamma measure is a plain periodogram mean over 30-80 Hz;
the data set has no day labels.

What was found about the data set: rat 3's home-cage session 2 is a second deposit of 287 segments
of its session 1; the files hold more removed segments (2.6% to 23.5%) than the SI's 1-4%; each file
has its own clock, so no recording spans the transfer between arena and home cage; the LFP unit is
not stated; movement values differ in scale between rats; the descriptor names `Hcage` where files
say `HCage`, and calls the archive a file to unpack where it is a gzipped tar; the data set holds
only the BF LFP of six rats, not the ACC, VC, unit, object or behaviour data of the paper.